In [ ]:
#| include: false
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Markdown

pio.renderers.default = 'notebook'
pio.templates.default = 'plotly_white'

from aavolve.report_helpers import *

# parameters (overridden by papermill)
group_id = 'GROUP'
manifest = 'out/qc/group_reports/GROUP_manifest.tsv'


<script type="text/javascript">
function aavolveResizePlotly(container) {
  if (typeof require === "undefined") return;
  require(["plotly"], function(Plotly) {
    (container || document).querySelectorAll(".plotly-graph-div").forEach(function(gd) {
      try { Plotly.Plots.resize(gd); } catch (e) {}
    });
  });
}
document.addEventListener("shown.bs.tab", function(e) {
  var selector = e.target && e.target.getAttribute && e.target.getAttribute("data-bs-target");
  var pane = selector ? document.querySelector(selector) : null;
  aavolveResizePlotly(pane);
});
window.addEventListener("resize", function() { aavolveResizePlotly(); });
</script>


In [ ]:
#| include: false
manifest_df = pd.read_csv(manifest, sep='	')
samples = manifest_df['sample'].tolist()
parent_file = manifest_df['parent_file'].iloc[0]
reference_file = manifest_df['reference_file'].iloc[0]
color_map = parent_colors_for_group(manifest_df['parent_frequencies'].tolist())


In [ ]:
display(Markdown('This report aggregates all samples which share the same parent/reference combination.'))
display(Markdown(f'- Group ID: `{group_id}`'))
display(Markdown(f'- Parent file: `{parent_file}`'))
display(Markdown(f'- Reference file: `{reference_file}`'))
display(Markdown(f"- Samples: `{', '.join(samples)}`"))


In [ ]:
#| echo: false
parents_warn = manifest_df["parents_dropped_warning"].iloc[0] if "parents_dropped_warning" in manifest_df.columns else ""
variant_warn = manifest_df["variant_window_warning"].iloc[0] if "variant_window_warning" in manifest_df.columns else ""
display_warning_file(parents_warn, title="Warning")
display_warning_file(variant_warn, title="Warning")


In [ ]:
#| title: Read counts at each processing stage (all samples)

dfs = []
for row in manifest_df.itertuples(index=False):
    df = import_read_count_data(row.read_counts, row.seq_tech)
    df = df[df['File type'] != 'Distinct at nucleotide level']
    df = df[df['File type'] != 'Distinct at amino acid level']
    df['sample'] = row.sample
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

fig = px.line(df_all, x='File type', y='Count', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: Fraction of reads retained after each filter (all samples)

fig = px.line(df_all, x='File type', y='Fraction of reads', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: QC summary table

from IPython.display import display
rows = []
for row in manifest_df.itertuples(index=False):
    # reads passing all filters (count + percent of input)
    try:
        n_pass, frac_pass = reads_passing_all_filters(row.read_counts, row.seq_tech)
    except Exception:
        n_pass, frac_pass = 0, 0.0
    # coverage depth summary
    cov_min = cov_median = cov_low = None
    if hasattr(row, "coverage_depth") and isinstance(row.coverage_depth, str) and row.coverage_depth:
        try:
            cov = coverage_depth_summary(row.coverage_depth)
            cov_min = cov["min"]
            cov_median = cov["median"]
            cov_low = cov["low_frac"]
        except Exception:
            pass
    rows.append({
        "sample": row.sample,
        "reads_passing_all": n_pass,
        "pct_passing_all": frac_pass*100,
        "coverage_min": cov_min,
        "coverage_median": cov_median,
        "coverage_low_%(<10)": None if cov_low is None else cov_low*100,
        "trim": getattr(row, "trim", None),
        "include_non_parental": getattr(row, "include_non_parental", None),
    })
display(pd.DataFrame(rows))


In [ ]:
#| title: Variant grouping QC (per sample)

from IPython.display import display, Markdown
import math

rows = []
for row in manifest_df.itertuples(index=False):
    # read drop-off at grouping stage
    try:
        drop = grouping_drop_stats(row.read_counts, row.seq_tech)
    except Exception:
        drop = {"refcov_reads": None, "pivoted_reads": None, "dropped_reads": None, "dropped_fraction": None}

    group_vars = getattr(row, "group_vars", False)
    group_dist = getattr(row, "group_vars_dist", "")
    max_frac = getattr(row, "max_group_distance", "")

    combined_variants = f"out/variants/combined/{row.sample}.tsv.gz"

    n_groups = None
    median_group_size = None
    max_group_size = None
    min_group_size = None
    n_exact_groups = None
    pct_exact_groups = None
    min_size_for_1_mismatch = None

    try:
        groups_df = variant_grouping_table(combined_variants, group_vars, group_dist, max_frac)
        n_groups = int(len(groups_df))
        if n_groups:
            median_group_size = float(groups_df["group_size"].median())
            max_group_size = int(groups_df["group_size"].max())
            min_group_size = int(groups_df["group_size"].min())
            n_exact_groups = int((groups_df["max_allowed_mismatches"] == 0).sum())
            pct_exact_groups = (n_exact_groups / n_groups) * 100

            effective_max_frac = float(groups_df["max_distance_frac"].iloc[0]) if "max_distance_frac" in groups_df.columns else 0.0
            if effective_max_frac > 0:
                min_size_for_1_mismatch = int(math.ceil(1.0 / effective_max_frac))
            else:
                min_size_for_1_mismatch = None
    except Exception:
        pass

    rows.append(
        {
            "sample": row.sample,
            "group_vars": group_vars,
            "group_dist": group_dist,
            "max_distance_frac": max_frac,
            "min_group_size_for_1_mismatch": min_size_for_1_mismatch,
            "n_groups": n_groups,
            "median_group_size": median_group_size,
            "min_group_size": min_group_size,
            "max_group_size": max_group_size,
            "exact_match_groups_%": pct_exact_groups,
            "refcov_reads": drop["refcov_reads"],
            "pivoted_reads": drop["pivoted_reads"],
            "dropped_at_grouping_reads": drop["dropped_reads"],
            "dropped_at_grouping_%": None if drop["dropped_fraction"] is None else drop["dropped_fraction"] * 100,
        }
    )

df = pd.DataFrame(rows)
display(df)

try:
    flagged = df[df["dropped_at_grouping_%"].fillna(0) >= 50][["sample", "dropped_at_grouping_%"]]
    if len(flagged) > 0:
        display(Markdown("**Warning**: substantial read drop at grouping stage (>= 50%):"))
        display(flagged)
except Exception:
    pass


In [ ]:
#| title: Trimming overhang summary (pre/post)

from IPython.display import display
rows = []
for row in manifest_df.itertuples(index=False):
    pre_frac = post_frac = None
    try:
        df_pre, _ = msa_overhangs(row.pretrim_msa)
        pre_frac = float(((df_pre.overhang_5 > 5) | (df_pre.overhang_3 > 5)).mean())
    except Exception:
        pass
    post_path = getattr(row, "posttrim_msa", "")
    if isinstance(post_path, str) and post_path:
        try:
            df_post, _ = msa_overhangs(post_path)
            post_frac = float(((df_post.overhang_5 > 5) | (df_post.overhang_3 > 5)).mean())
        except Exception:
            pass
    rows.append({"sample": row.sample, "pre_overhang_%(>5bp)": None if pre_frac is None else pre_frac*100, "post_overhang_%(>5bp)": None if post_frac is None else post_frac*100})
display(pd.DataFrame(rows))


In [ ]:
#| title: Overall parent assignment summary (mean across variants)

summary_rows = []
for row in manifest_df.itertuples(index=False):
    df = pd.read_csv(row.parent_frequencies, sep='	')
    df = df[~df['parent'].astype(str).str.match('non_parental_\\d+')]
    df['parent'] = df['parent'].astype(str).str.replace('non_parental_\d+', 'non parental', regex=True)
    df_sum = df.groupby('parent', as_index=False)['frequency'].mean()
    df_sum['frequency'] = df_sum['frequency'] * 100
    df_sum['sample'] = row.sample
    summary_rows.append(df_sum)
summary_df = pd.concat(summary_rows, ignore_index=True)

fig = px.bar(
    summary_df,
    x='parent',
    y='frequency',
    color='parent',
    facet_col='sample',
    facet_col_wrap=2,
    labels={'frequency': 'Mean parent frequency (%)', 'parent': 'Parent'},
    color_discrete_map=color_map,
)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(margin=dict(l=60, r=40, t=60, b=160), showlegend=False)
fig

In [ ]:
#| title: Non-parental variants (group summary)

from IPython.display import display, Markdown


def _fmt_variant(pos, ref, alt):
    pos = str(pos)
    ref = str(ref)
    alt = str(alt)
    if ref == ".":
        return f"{pos}ins{alt}"
    if alt == ".":
        return f"{ref}{pos}del"
    return f"{ref}{pos}{alt}"


rows = []
for row in manifest_df.itertuples(index=False):
    include = str(getattr(row, "include_non_parental", "False")) == "True"
    if not include:
        continue

    thr = float(getattr(row, "non_parental_freq", 0.0) or 0.0)

    try:
        df = pd.read_csv(row.variant_freq_all, sep="	")
    except Exception:
        continue

    df = df[df.query_name == "non_parental"].copy()
    if len(df) == 0:
        continue

    df["freq"] = pd.to_numeric(df["freq"], errors="coerce")
    df = df.dropna(subset=["freq"])
    if len(df) == 0:
        continue

    try:
        refcov = get_read_count(row.read_counts, row.seq_tech, "Filtered by reference coverage")
    except Exception:
        refcov = 0

    df["sample"] = row.sample
    df["threshold"] = thr
    df["refcov"] = int(refcov)
    df["variant"] = df.apply(lambda r: _fmt_variant(r.pos, r.ref_bases, r.query_bases), axis=1)
    rows.append(df[["sample", "threshold", "refcov", "variant", "freq"]])


if not rows:
    display(Markdown("No non-parental variants included in this group."))
else:
    all_np_df = pd.concat(rows, ignore_index=True)

    included_np_df = all_np_df[all_np_df["freq"] >= all_np_df["threshold"]].copy()
    if len(included_np_df) == 0:
        display(Markdown("No non-parental variants included in this group."))
    else:
        included_variants = sorted(included_np_df["variant"].unique())

        sample_order = [
            r.sample
            for r in manifest_df.itertuples(index=False)
            if str(getattr(r, "include_non_parental", "False")) == "True"
        ]
        thr_by_sample = (
            all_np_df.drop_duplicates("sample")[["sample", "threshold"]]
            .set_index("sample")["threshold"]
            .to_dict()
        )
        refcov_by_sample = (
            all_np_df.drop_duplicates("sample")[["sample", "refcov"]]
            .set_index("sample")["refcov"]
            .to_dict()
        )
        total_reads = int(sum(refcov_by_sample.get(s, 0) for s in sample_order))

        included_np_df["read_count"] = (included_np_df["freq"] * included_np_df["refcov"]).round().astype(int)

        agg = included_np_df.groupby("variant", as_index=False).agg(
            n_samples=("sample", "nunique"),
            samples=("sample", lambda xs: ", ".join(sorted(set(xs)))),
            total_read_count=("read_count", "sum"),
        )
        if total_reads > 0:
            agg["total_read_fraction"] = agg["total_read_count"] / total_reads
        else:
            agg["total_read_fraction"] = 0.0

        freq_wide = (
            all_np_df[all_np_df["variant"].isin(included_variants)]
            .pivot_table(index="variant", columns="sample", values="freq", aggfunc="max")
            .reindex(columns=sample_order)
        )

        colname_by_sample = {
            s: f"{s} (thr={thr_by_sample.get(s, 0.0):g})" for s in sample_order
        }
        freq_wide = freq_wide.rename(columns=colname_by_sample)

        out = agg.merge(freq_wide.reset_index(), on="variant", how="left")
        out = out.sort_values(["total_read_fraction", "total_read_count"], ascending=[False, False]).reset_index(drop=True)

        per_sample_cols = [colname_by_sample[s] for s in sample_order if colname_by_sample[s] in out.columns]

        def _bold_passing_threshold(data):
            styles = pd.DataFrame("", index=data.index, columns=data.columns)
            for sample in sample_order:
                col = colname_by_sample.get(sample)
                if col not in data.columns:
                    continue
                thr = thr_by_sample.get(sample, 0.0)
                mask = data[col].notna() & (data[col] >= thr)
                styles.loc[mask, col] = "font-weight: bold"
            return styles

        styler = out.style
        if per_sample_cols:
            styler = styler.apply(_bold_passing_threshold, axis=None, subset=per_sample_cols)

        styler = styler.format(
            {
                "total_read_fraction": "{:.6f}",
                **{c: "{:.6f}" for c in per_sample_cols},
            },
            na_rep="",
        )

        display(styler)


In [ ]:
#| title: Assigned parents for top reads (per sample)
#| echo: false

from IPython.display import HTML, display
import json
from html import escape
from plotly.utils import PlotlyJSONEncoder

PLOT_HEIGHT = 650
SCROLL_HEIGHT = 900

css = (
    '<style>'
    '.aavolve-scroll-container,\n'
    '.aavolve-scroll-container.html-fill-container,\n'
    '.aavolve-scroll-container.html-fill-item {\n'
    '  height: ' + str(SCROLL_HEIGHT) + 'px !important;\n'
    '  max-height: ' + str(SCROLL_HEIGHT) + 'px !important;\n'
    '  overflow-y: auto !important;\n'
    '  display: block !important;\n'
    '  flex: 0 0 auto !important;\n'
    '}\n'
    '.aavolve-heatmap-wrap,\n'
    '.aavolve-heatmap-wrap.html-fill-container,\n'
    '.aavolve-heatmap-wrap.html-fill-item {\n'
    '  height: ' + str(PLOT_HEIGHT) + 'px !important;\n'
    '  min-height: ' + str(PLOT_HEIGHT) + 'px !important;\n'
    '  width: 100% !important;\n'
    '  display: block !important;\n'
    '  flex: 0 0 auto !important;\n'
    '}\n'
    '.aavolve-heatmap-legend {\n'
    '  width: 100% !important;\n'
    '  display: flex !important;\n'
    '  flex-direction: row !important;\n'
    '  align-items: center !important;\n'
    '  gap: 12px;\n'
    '  text-align: left !important;\n'
    '  margin: 0 0 10px 0;\n'
    '  padding: 8px 10px;\n'
    '  border: 1px solid #e5e7eb;\n'
    '  border-radius: 6px;\n'
    '  background: rgba(255,255,255,0.95);\n'
    '}\n'
    '.aavolve-heatmap-legend-title { font-weight: 600; margin-right: 6px; }\n'
    '.aavolve-heatmap-legend-items {\n'
    '  display: flex !important;\n'
    '  flex-direction: row !important;\n'
    '  flex-wrap: wrap !important;\n'
    '  justify-content: center !important;\n'
    '  align-content: flex-start !important;\n'
    '  gap: 10px 14px;\n'
    '  row-gap: 10px;\n'
    '  column-gap: 14px;\n'
    '  flex: 1 1 auto !important;\n'
    '  min-width: 0;\n'
    '}\n'
    '.aavolve-legend-item { display: inline-flex !important; align-items: center !important; gap: 8px; font-size: 12px; flex: 0 0 auto !important; }\n'
    '.aavolve-legend-swatch { width: 12px; height: 12px; border: 1px solid rgba(0,0,0,0.25); border-radius: 3px; flex: 0 0 12px; }\n'
    '.aavolve-legend-label { white-space: nowrap; }\n'
    '</style>'
)

blocks = []
blocks.append(css)
blocks.append(
    f"<div class='aavolve-scroll-container' style='border: 1px solid #e5e7eb; border-radius: 6px; padding: 10px;'>"
)

# Build a single horizontal legend from the shared group color_map so colors are consistent.
legend_items = []
legend_items.append("<div class='aavolve-heatmap-legend'>")
legend_items.append("<span class='aavolve-heatmap-legend-title'>Parents</span>")
legend_items.append("<div class='aavolve-heatmap-legend-items'>")
for parent, color in color_map.items():
    legend_items.append(
        f"<div class='aavolve-legend-item'><span class='aavolve-legend-swatch' style='background:{escape(str(color))}'></span><span class='aavolve-legend-label'>{escape(str(parent))}</span></div>"
    )
legend_items.append("</div>")
legend_items.append("</div>")
legend_html = ''.join(legend_items)

for i, row in enumerate(manifest_df.itertuples(index=False)):
    blocks.append(f"<h3 style='margin: 0 0 10px 0;'>{escape(str(row.sample))}</h3>")

    fig = parent_heatmap(row.assigned_parents, row.parent_frequencies, color_dict=color_map, reserve_legend_space=False)
    if fig is None:
        blocks.append("<div style='margin: 0 0 18px 0; color: #6b7280;'>No data</div>")
        continue

    fig.update_layout(height=PLOT_HEIGHT, showlegend=False)

    div_id = f"parent-heatmap-{i}"
    blocks.append(legend_html)
    blocks.append(f"<div id='{div_id}' class='aavolve-heatmap-wrap'></div>")

    fig_payload = json.dumps(fig.to_plotly_json(), cls=PlotlyJSONEncoder)
    blocks.append(
        "<script>"
        f"require(['plotly'], function(Plotly) {{ var fig = {fig_payload}; Plotly.newPlot('{div_id}', fig.data, fig.layout, {{responsive: true}}); }});"
        "</script>"
    )

blocks.append("</div>")
display(HTML("\n".join(blocks)))


In [ ]:
#| title: Mean pairwise distance between top capsids

metric_rows = []
for row in manifest_df.itertuples(index=False):
    for label, path in [
        ('nt-first', row.dmat_nt_first),
        ('aa-first', row.dmat_aa_first),
    ]:
        dmat = np.loadtxt(path, ndmin=2)
        mask = ~np.eye(dmat.shape[0], dtype=bool)
        mean_distance = 0.0 if mask.sum() == 0 else float(dmat[mask].mean())
        metric_rows.append({'sample': row.sample, 'matrix': label, 'mean_distance': mean_distance})
metrics_df = pd.DataFrame(metric_rows)

fig = px.bar(metrics_df, x='sample', y='mean_distance', color='matrix', barmode='group',
             labels={'mean_distance': 'Mean pairwise distance', 'sample': 'Sample', 'matrix': 'Matrix'})
fig.update_layout(margin=dict(l=60, r=40, t=60, b=80))
fig

In [ ]:
#| title: Settings (per sample)

cols = [c for c in [
    "sample", "seq_tech", "trim", "adapter_5", "adapter_3", "anchors",
    "require_end_to_end_alignment", "include_non_parental", "non_parental_freq",
    "group_vars", "group_vars_dist", "max_group_distance", "minimap2_params"
] if c in manifest_df.columns]
display(manifest_df[cols])
